# 2. Profile Dataset

Profile a logged immutable dataset from MLflow/GCS artifacts. This notebook does not rebuild the data pipeline, refresh source data, or derive a train/test split.

## 1. Load Project

In [1]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

import automl
from automl import data

DRY_RUN = True


In [2]:
active = automl.use_project(dry_run=DRY_RUN)
config = active.config
display(
    {
        "project": active.project_name,
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
        "experiment": active.active_experiment_id,
        "dry_run": active.dry_run,
    }
)


{'project': 'example_homecredit',
 'repo_root': '/Users/wendao/1.working_directory/automl',
 'project_dir': '/Users/wendao/1.working_directory/automl/projects/example_homecredit',
 'experiment': 'example-homecredit',
 'dry_run': True}

## 2. List Logged Datasets

In [3]:
index = data.list_datasets(session=active)
index.to_dataframe()


,schema_version,id,identity_hash,component_hashes,gcs_bucket,gcs_prefix,project_name,created_at,source_identity,n_rows,n_columns,target_column,split_id_col,hash_key,data_gcs_uri,registry_gcs_uri,manifest_gcs_uri
0,1,v1_51b1c03a,sha256:51b1c03a9d267c7e01926685057a82990b9a1c7...,{'source_identity': 'sha256:11d9b27b6304ded71c...,data_science_test_remote,automl/dry_run,example_homecredit,2026-06-02T17:02:56.428282+00:00,"{'kind': 'local_csv', 'csv_path': '/Users/wend...",100,108,target,SPLITID,[sk_id_curr],gs://data_science_test_remote/automl/dry_run/e...,gs://data_science_test_remote/automl/dry_run/e...,gs://data_science_test_remote/automl/dry_run/e...


## 3. Select Dataset Explicitly

The default is the active dataset. Edit `DATASET_ID` if you want a different logged dataset.

In [4]:
DATASET_ID = index.active_dataset_id or (index.datasets[-1].id if index.datasets else None)
DATASET_ID


'v1_51b1c03a'

## 4. Load Dataset From Artifacts

This reads the full dataset dataframe, feature registry, and manifest from logged artifacts. It does not call the data builder.

In [5]:
if DATASET_ID is None:
    raise RuntimeError("No logged dataset is available. Run materialization first.")

loaded_dataset = data.load_dataset_by_id(DATASET_ID, session=active)
loaded_dataset.dataset.id


'v1_51b1c03a'

## 5. Dataset Sanity Views

In [6]:
loaded_dataset.df.head()


,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,flag_document_14,flag_document_16,flag_document_18,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year,SPLITID
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,85
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,62
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,3
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,41
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,74


In [7]:
loaded_dataset.df.shape


(100, 108)

In [8]:
loaded_dataset.registry.to_dataframe()


,name,dtype,original_name,null_pct,nunique,dominance_pct,available,feature,model,target,comments,derived,source_columns
0,sk_id_curr,num,SK_ID_CURR,0.00,100,0.01,True,False,False,False,,False,[]
1,target,num,TARGET,0.00,2,0.94,True,False,False,True,,False,[]
2,name_contract_type,cat,NAME_CONTRACT_TYPE,0.00,2,0.85,True,True,True,False,,False,[]
3,code_gender,cat,CODE_GENDER,0.00,2,0.57,True,True,True,False,,False,[]
4,flag_own_car,cat,FLAG_OWN_CAR,0.00,2,0.70,True,True,True,False,,False,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,amt_req_credit_bureau_week,num,AMT_REQ_CREDIT_BUREAU_WEEK,0.15,2,0.83,True,True,True,False,,False,[]
104,amt_req_credit_bureau_mon,num,AMT_REQ_CREDIT_BUREAU_MON,0.15,4,0.70,True,True,True,False,,False,[]
105,amt_req_credit_bureau_qrt,num,AMT_REQ_CREDIT_BUREAU_QRT,0.15,4,0.69,True,True,True,False,,False,[]
106,amt_req_credit_bureau_year,num,AMT_REQ_CREDIT_BUREAU_YEAR,0.15,7,0.31,True,True,True,False,,False,[]


In [9]:
loaded_dataset.dataset.to_dict()


{'schema_version': 1,
 'id': 'v1_51b1c03a',
 'identity_hash': 'sha256:51b1c03a9d267c7e01926685057a82990b9a1c74fb4759eda3f9b77d8875f1fd',
 'component_hashes': {'source_identity': 'sha256:11d9b27b6304ded71c9fee78bd717d839a2bb40e27f34fff6b31461a93b39b02',
  'feature_registry': 'sha256:5153ea24d069ff3a97e483ec7891b27caa3101eeeda4e9647d6ad5c7c7a31255',
  'data_content': 'sha256:dcdc36389dbfe7a30b130ebe81cd4837cc04ac388ddadf484b07136c6a6591d4',
  'schema': 'sha256:8b7a337cb399ab0d2b66984a0882bcabc8783f321b0f581fc1535386a4b8e380'},
 'gcs_bucket': 'data_science_test_remote',
 'gcs_prefix': 'automl/dry_run',
 'project_name': 'example_homecredit',
 'created_at': '2026-06-02T17:02:56.428282+00:00',
 'source_identity': {'kind': 'local_csv',
  'csv_path': '/Users/wendao/1.working_directory/automl/projects/example_homecredit/data/application_train_sample.csv',
  'hash_key': ['sk_id_curr']},
 'n_rows': 100,
 'n_columns': 108,
 'target_column': 'target',
 'split_id_col': 'SPLITID',
 'hash_key': ['sk_i

In [10]:
target_col = loaded_dataset.dataset.target_column
loaded_dataset.df[target_col].value_counts(dropna=False).to_frame("rows")


,rows
target,
0,94
1,6


In [11]:
loaded_dataset.df.isna().mean().sort_values(ascending=False).to_frame("null_rate").head(30)


,null_rate
commonarea_avg,0.73
commonarea_medi,0.73
commonarea_mode,0.73
nonlivingapartments_avg,0.70
nonlivingapartments_medi,0.70
own_car_age,0.70
nonlivingapartments_mode,0.70
livingapartments_medi,0.70
livingapartments_avg,0.70
livingapartments_mode,0.70


In [12]:
loaded_dataset.df.dtypes.astype(str).value_counts().to_frame("columns")


,columns
float64,65
int64,27
object,16


In [13]:
categorical_cols = loaded_dataset.df.select_dtypes(include=["object", "category"]).columns
loaded_dataset.df[categorical_cols].nunique(dropna=True).sort_values(ascending=False).to_frame("cardinality").head(30)


,cardinality
organization_type,32
occupation_type,14
weekday_appr_process_start,7
wallsmaterial_mode,6
name_type_suite,5
name_family_status,5
name_income_type,4
name_housing_type,4
name_education_type,3
fondkapremont_mode,3


In [14]:
loaded_dataset.registry.to_dataframe()[["available", "feature", "target", "model"]].sum().to_frame("columns")


,columns
available,108
feature,105
target,1
model,105


In [15]:
{
    "source_identity": loaded_dataset.dataset.source_identity,
    "component_hashes": loaded_dataset.dataset.component_hashes.to_dict(),
}


{'source_identity': {'kind': 'local_csv',
  'csv_path': '/Users/wendao/1.working_directory/automl/projects/example_homecredit/data/application_train_sample.csv',
  'hash_key': ['sk_id_curr']},
 'component_hashes': {'source_identity': 'sha256:11d9b27b6304ded71c9fee78bd717d839a2bb40e27f34fff6b31461a93b39b02',
  'feature_registry': 'sha256:5153ea24d069ff3a97e483ec7891b27caa3101eeeda4e9647d6ad5c7c7a31255',
  'data_content': 'sha256:dcdc36389dbfe7a30b130ebe81cd4837cc04ac388ddadf484b07136c6a6591d4',
  'schema': 'sha256:8b7a337cb399ab0d2b66984a0882bcabc8783f321b0f581fc1535386a4b8e380'}}

## 6. Ad Hoc Exploration Beyond The Standard Profile

These cells display investigation tables for a human reviewer. They do not mutate `config.py`.

In [16]:
hash_key = list(loaded_dataset.dataset.hash_key)
hash_key_health = pd.Series(
    {
        "hash_key": hash_key,
        "duplicate_key_rows": int(loaded_dataset.df.duplicated(subset=hash_key, keep=False).sum()) if hash_key else None,
        "null_key_rows": int(loaded_dataset.df[hash_key].isna().any(axis=1).sum()) if hash_key else None,
    }
)
hash_key_health


hash_key              [sk_id_curr]
duplicate_key_rows               0
null_key_rows                    0
dtype: object

In [17]:
loaded_dataset.df.loc[loaded_dataset.df.duplicated(subset=hash_key, keep=False), hash_key].head(20) if hash_key else pd.DataFrame()


,sk_id_curr


In [18]:
full_row_duplication = pd.Series(
    {
        "duplicate_rows": int(loaded_dataset.df.duplicated(keep=False).sum()),
        "duplicate_rate": float(loaded_dataset.df.duplicated(keep=False).mean()),
    }
)
full_row_duplication


duplicate_rows    0.0
duplicate_rate    0.0
dtype: float64

In [19]:
split_id_col = loaded_dataset.dataset.split_id_col
(
    loaded_dataset.df[split_id_col]
    .value_counts(dropna=False)
    .sort_index()
    .reindex(range(100), fill_value=0)
    .to_frame("rows")
    if split_id_col in loaded_dataset.df.columns
    else pd.DataFrame()
)


,rows
SPLITID,
0,1
1,1
2,0
3,2
4,1
...,...
95,0
96,0
97,3


In [20]:
target_missing = loaded_dataset.df[target_col].isna()
{
    "missing_target_rows": int(target_missing.sum()),
    "examples": loaded_dataset.df.loc[target_missing].head(10),
}


{'missing_target_rows': 0,
 'examples': Empty DataFrame
 Columns: [sk_id_curr, target, name_contract_type, code_gender, flag_own_car, flag_own_realty, cnt_children, amt_income_total, amt_credit, amt_annuity, amt_goods_price, name_type_suite, name_income_type, name_education_type, name_family_status, name_housing_type, region_population_relative, days_birth, days_employed, days_registration, days_id_publish, own_car_age, flag_emp_phone, flag_work_phone, flag_cont_mobile, flag_phone, flag_email, occupation_type, cnt_fam_members, region_rating_client, region_rating_client_w_city, weekday_appr_process_start, hour_appr_process_start, reg_region_not_live_region, reg_region_not_work_region, live_region_not_work_region, reg_city_not_live_city, reg_city_not_work_city, live_city_not_work_city, organization_type, ext_source_1, ext_source_2, ext_source_3, apartments_avg, basementarea_avg, years_beginexpluatation_avg, years_build_avg, commonarea_avg, elevators_avg, entrances_avg, floorsmax_avg, flo

In [21]:
missing_by_target = []
if loaded_dataset.df[target_col].nunique(dropna=True) <= 20:
    for col in loaded_dataset.df.columns:
        if col == target_col:
            continue
        rates = loaded_dataset.df.assign(_missing=loaded_dataset.df[col].isna()).groupby(target_col, dropna=False)["_missing"].mean()
        if len(rates) > 1 and rates.max() - rates.min() >= 0.25:
            missing_by_target.append({"column": col, "min_null_rate": rates.min(), "max_null_rate": rates.max()})
pd.DataFrame(missing_by_target).sort_values("max_null_rate", ascending=False) if missing_by_target else pd.DataFrame()


""


In [22]:
rare_categories = []
for col in categorical_cols:
    counts = loaded_dataset.df[col].value_counts(dropna=False)
    tiny = counts[counts <= max(2, len(loaded_dataset.df) * 0.005)]
    if len(tiny):
        rare_categories.append({"column": col, "rare_category_count": len(tiny), "examples": tiny.head(5).to_dict()})
pd.DataFrame(rare_categories).sort_values("rare_category_count", ascending=False) if rare_categories else pd.DataFrame()


,column,rare_category_count,examples
4,organization_type,22,"{'Business Entity Type 1': 2, 'Services': 2, '..."
3,occupation_type,9,"{'Accountants': 2, 'Cooking staff': 2, 'Privat..."
0,name_type_suite,4,"{'Spouse, partner': 2, 'Children': 2, 'Other_A..."
5,wallsmaterial_mode,3,"{'Others': 2, 'Mixed': 1, 'Wooden': 1}"
2,name_housing_type,2,"{'Rented apartment': 2, 'Municipal apartment': 2}"
1,name_education_type,1,{'Incomplete higher': 2}
6,emergencystate_mode,1,{'Yes': 1}


In [23]:
high_cardinality = (
    loaded_dataset.df[categorical_cols]
    .nunique(dropna=True)
    .sort_values(ascending=False)
    .to_frame("cardinality")
)
high_cardinality[high_cardinality["cardinality"] > 50]


,cardinality


In [24]:
near_constant = []
for col in loaded_dataset.df.columns:
    counts = loaded_dataset.df[col].value_counts(dropna=False, normalize=True)
    if len(counts) <= 1 or (len(counts) and counts.iloc[0] >= 0.995):
        near_constant.append({"column": col, "dominant_share": float(counts.iloc[0]) if len(counts) else 0.0})
pd.DataFrame(near_constant).sort_values("dominant_share", ascending=False) if near_constant else pd.DataFrame()


""


In [25]:
datetime_review = []
for col in loaded_dataset.df.columns:
    if "DATE" in col.upper() or "TIME" in col.upper():
        parsed = pd.to_datetime(loaded_dataset.df[col], errors="coerce")
        datetime_review.append(
            {
                "column": col,
                "parse_failure_rate": float(parsed.isna().mean()),
                "min": parsed.min(),
                "max": parsed.max(),
                "future_rows": int((parsed > pd.Timestamp.utcnow().tz_localize(None)).sum()) if parsed.notna().any() else 0,
            }
        )
pd.DataFrame(datetime_review) if datetime_review else pd.DataFrame()


""


In [26]:
numeric = loaded_dataset.df.select_dtypes(include="number")
outlier_scan = numeric.quantile([0, 0.001, 0.01, 0.5, 0.99, 0.999, 1.0]).T
outlier_scan.head(50)


,0.000,0.001,0.010,0.500,0.990,0.999,1.000
sk_id_curr,100002.000000,100002.099000,100002.990000,100057.000000,1.001160e+05,1.001178e+05,1.001180e+05
target,0.000000,0.000000,0.000000,0.000000,1.000000e+00,1.000000e+00,1.000000e+00
cnt_children,0.000000,0.000000,0.000000,0.000000,3.000000e+00,3.000000e+00,3.000000e+00
amt_income_total,38419.155000,39961.658655,53844.191550,135000.000000,5.400000e+05,5.400000e+05,5.400000e+05
amt_credit,80865.000000,82357.425000,95789.250000,496435.500000,1.561759e+06,1.653765e+06,1.663988e+06
amt_annuity,5301.000000,5358.469500,5875.695000,24646.500000,6.433583e+04,8.472413e+04,8.698950e+04
amt_goods_price,67500.000000,69727.500000,89775.000000,452250.000000,1.530585e+06,1.582709e+06,1.588500e+06
region_population_relative,0.003069,0.003074,0.003121,0.019101,7.250800e-02,7.250800e-02,7.250800e-02
days_birth,-24827.000000,-24823.733000,-24794.330000,-15388.500000,-7.973330e+03,-7.913633e+03,-7.907000e+03
days_employed,-9523.000000,-9457.561000,-8868.610000,-1320.500000,3.652430e+05,3.652430e+05,3.652430e+05


In [27]:
leakage_terms = ("TARGET", "LABEL", "OUTCOME", "STATUS", "DEFAULT", "PAID", "APPROVED")
name_based_suspects = [
    col for col in loaded_dataset.df.columns
    if col != target_col and any(term in col.upper() for term in leakage_terms)
]
name_based_suspects


['name_family_status']

In [28]:
registry_df = loaded_dataset.registry.to_dataframe()
registry_columns = set(registry_df["name"])
data_columns = set(loaded_dataset.df.columns)
registry_mismatches = {
    "data_missing_from_registry": sorted(data_columns - registry_columns),
    "registry_missing_from_data": sorted(registry_columns - data_columns),
    "target_rows": registry_df[registry_df["target"] == True],
    "metadata_or_excluded_target_conflicts": registry_df[(registry_df["target"] == True) & (registry_df["feature"] == True)],
}
registry_mismatches


{'data_missing_from_registry': [],
 'registry_missing_from_data': [],
 'target_rows':      name dtype original_name  null_pct  nunique  dominance_pct  available  \
 1  target   num        TARGET       0.0        2           0.94       True   
 
    feature  model  target comments  derived source_columns  
 1    False  False    True             False             []  ,
 'metadata_or_excluded_target_conflicts': Empty DataFrame
 Columns: [name, dtype, original_name, null_pct, nunique, dominance_pct, available, feature, model, target, comments, derived, source_columns]
 Index: []}

## 7. Run Deterministic Profile

This writes local profile artifacts and logs them to the experiment overview run using `MlflowClient` with an explicit run id.

In [ ]:
profile_result = data.profile(dataset_id=DATASET_ID, session=active)
pd.DataFrame([profile_result.to_dict()])


## 8. MLflow Artifact Logging Contract

Expected logged artifacts:

- `{DATASET_ID}/profile/data_card.json`
- `{DATASET_ID}/profile/data_observations.json`
- `{DATASET_ID}/profile/profile_manifest.json`
- `{DATASET_ID}/profile/charts/...`

In [ ]:
{
    "data_card_uri": profile_result.data_card_uri,
    "data_observations_uri": profile_result.data_observations_uri,
    "profile_manifest_uri": profile_result.profile_manifest_uri,
    "chart_uris": profile_result.chart_uris,
}


## 9. Inspect Logged Profile Artifacts

In [ ]:
pd.DataFrame([profile_result.to_dict()])
